### LIBRARIES

In [ ]:
import os
import re
import sys
import time

import anthropic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score, f1_score

In [ ]:
columns= {

'age_at_visit': 'Age (years)',
'gender': 'Gender', 
'disease_duration': 'Disease duration (years)',

'rf_status': 'RF status',
'anti_ccp_status': 'Anti-CCP status',

'das28_score': 'DAS28 score',
'das28_change': 'DAS28 change',

'crp': 'CRP (mg/L)',
'crp_upper_limit': 'CRP upper limit of normal (mg/L)',
'esr': 'ESR (mm/hr)',

'mtx_dose': 'MTX dose (mg/week)',
' steroid_dose': 'Steroid dose (mg/day)',
'csdmard_use': 'csDMARD use',
'b_ts_dmard_use': 'b/tsDMARD use',
'treatment_decision_grouped': 'Treatment decision',

'previous_flare': 'Flare at previous visit',
'days_since_last_visit': 'Days since last visit',
'comorbidities': 'Comorbidities',
}

In [ ]:
df_llm= df_cleaned.copy()

llm_feature_cols = [
    'patient_id',
    #Demographics
    'age_at_visit',
    'disease_duration',
    'gender',
    #Serology
    'rf_status',
    'anti_ccp_status',
    'das28_score',
    'das28_change',
    #Inflammatory markers
    'crp',
    'crp_upper_limit',
    'esr',
    #Treatment
    'mtx_dose',
    'steroid_dose',
    'csdmard_use',
    'b_ts_dmard_use',
    'treatment_decision',
    #History
    'previous_flare',
    'days_since_last_visit',
    #Comorbidities
    'comorbidities',
    #Target variable
    'flare_next_visit',
]

df_llm = df_llm[llm_feature_cols].copy()
print("Shape:", df_llm.shape)

In [ ]:
def get_das28_category(das28_score):
    if das28_score < 2.6:
        return 'remission'
    elif 2.6 <= das28_score < 3.2:
        return 'low disease activity'
    elif 3.2 <= das28_score < 5.1:
        return 'moderate disease activity'
    else:
        return 'high disease activity'
    
def get_crp_interpretation(crp, crp_upper_limit):
    if pd.isnull(crp) or pd.isnull(crp_upper_limit):
        return None
    ratio = crp / crp_upper_limit
    if ratio <= 1:
        return f"{crp:.2f} mg/L (within normal range; laboratory upper limit: {crp_upper_limit:.2f} mg/L)"
    else:
        return f"{crp:.2f} mg/L (elevated — {ratio:.1f}x the laboratory upper limit of {crp_upper_limit:.2f} mg/L)"

def get_gender_text(val):
    if pd.isna(val):
        return "Not recorded"
    return "Female" if val == 1 else "Male"
    
def get_binary_text(val, yes_label="Yes", no_label="No"):
    if pd.isna(val):
        return "Not recorded"
    return yes_label if val == 1 else no_label

def get_comorbidities_text(val):
    if pd.isna(val) or str(val).lower() == 'none':
        return "No known comorbidities"
    return str(val).replace(',', ', ')

def is_first_visit(row):
    return row['days_since_last_visit'] == 0

In [ ]:
#List Template
def serialize_list(row):
    """
    Structured list format for LLM input.
    """

    lines=[]

    #Demographics
    lines.append(f"Age: {row['age_at_visit']} years")
    
    if not pd.isna(row['disease_duration']):
        lines.append(f"Disease duration: {row['disease_duration']} years")
    else: 
        lines.append("Disease duration: Not recorded")
    lines.append(f"Gender: {get_gender_text(row['gender'])}")

    #Serology
    lines.append(f"RF status: {get_binary_text(row['rf_status'], 'Positive', 'Negative')}")
    lines.append(f"Anti-CCP status: {get_binary_text(row['anti_ccp_status'], 'Positive', 'Negative')}")

    #Disease activity
    if not pd.isna(row['das28_score']):
        das28_category = get_das28_category(row['das28_score'])
        lines.append(f"DAS28 score: {row['das28_score']:.2f} ({das28_category})")
    else:
        lines.append("DAS28 score: Not recorded at this visit.")

    #DAS28-change
    if not is_first_visit(row) and not pd.isna(row['das28_change']):
        direction = 'increased' if row['das28_change'] > 0 else \
                    'decreased' if row['das28_change'] < 0 else 'no change'
        lines.append(f"DAS28 change from previous visit: {row['das28_change']:.2f} ({direction})")

    #Inflammatory markers
    crp_text= get_crp_interpretation(row['crp'], row['crp_upper_limit'])
    if crp_text:
        lines.append(f"CRP: {crp_text}")
    else:
        lines.append("CRP: Not recorded at this visit")

    if not pd.isna(row['esr']):
        lines.append(f"ESR: {row['esr']:.1f} mm/hr")
    else: 
        lines.append("ESR: Not recorded at this visit")


    #Treatment
    lines.append(f"MTX dose: {row['mtx_dose']:.0f} mg/week"
                 if not pd.isna(row['mtx_dose'])  and row['mtx_dose'] > 0
                 else "MTX dose: Not in use at this visit")

    lines.append(f"Steroid dose: {row['steroid_dose']:.0f} mg"
                  if not pd.isna(row['steroid_dose']) and row['steroid_dose'] > 0
                  else "Steroid dose: Not in use at this visit")
    lines.append(f"csDMARD use: {get_binary_text(row['csdmard_use'])}")
    lines.append(f"b/tsDMARD use: {get_binary_text(row['b_ts_dmard_use'])}")
    decision_raw = row['treatment_decision']
    if decision_raw == 'no_decision':
        decision_clean = "No treatment adjustment made at this visit"
    else:
        decision_clean = str(decision_raw)
    lines.append(f"Recent treatment decision: {decision_clean}")


    #History
    if is_first_visit(row):
        lines.append("This is the patient's baseline clinic visit")
        lines.append("Flare at previous visit: Not applicable (no historical record available)")
    else:
        lines.append(f"Follow-up visit. Days since last assessment: {row['days_since_last_visit']} days")
        lines.append(f"Flare at previous visit: {get_binary_text(row['previous_flare'])}")

    #Comorbidities
    lines.append(f"Comorbidities: {get_comorbidities_text(row['comorbidities'])}")

    return '\n'.join(lines)

print(serialize_list(df_llm.iloc[5]))

In [ ]:
#Text Template
def serialize_text(row):
    """
    Structured text format for LLM input.
    """

    parts=[]

    #Demographics and serology
    if pd.isna(row['gender']):
        gender = "Gender not recorded"
    else:
        gender = 'female' if row['gender']==1 else 'male'

    duration = f"{row['disease_duration']} years" if not pd.isna(row['disease_duration']) else "unknown duration"

    rf='RF-positive' if (not pd.isna(row['rf_status']) and row['rf_status']==1)\
        else 'RF-negative' if (not pd.isna(row['rf_status']) and row['rf_status']==0)\
        else 'RF status unknown'
    anti_ccp='anti-CCP-positive' if (not pd.isna(row['anti_ccp_status']) and row['anti_ccp_status']==1)\
        else 'anti-CCP-negative' if (not pd.isna(row['anti_ccp_status']) and row['anti_ccp_status']==0)\
        else 'anti-CCP status unknown'
    
    age_text = f"{int(row['age_at_visit'])}" if not pd.isna(row['age_at_visit']) else "unknown age"
    parts.append(
        f"A {age_text}-year-old {gender} patient with "
        f"{duration} of rheumatoid arthritis, who is {rf} and {anti_ccp}."
    )

    #Disease activity
    if not pd.isna(row['das28_score']):
        das28_category = get_das28_category(row['das28_score'])
        das28_sentence = f"The DAS28 score at this visit is {row['das28_score']:.2f}, indicating {das28_category}."
        if not is_first_visit(row) and not pd.isna(row['das28_change']):
            direction = 'an increase' if row['das28_change'] > 0 \
                else 'a decrease' if row['das28_change'] < 0 else 'no change'
            das28_sentence += f" This represents {direction} of {abs(row['das28_change']):.2f} from the previous visit."
    else:
        das28_sentence = "The DAS28 score at this visit is not recorded."
    parts.append(das28_sentence)


    #Inflammation
    crp_text = get_crp_interpretation(row['crp'], row['crp_upper_limit'])
    esr_text = f"ESR is {row['esr']:.1f} mm/hr." if not pd.isna(row['esr']) else "ESR is not recorded at this visit." 

    if crp_text:
        parts.append(f"Inflammatory markers: CRP is {crp_text}. {esr_text}")
    else:
        parts.append(f"Inflammatory markers: CRP is not recorded at this visit. {esr_text}")


    #Treatment
    treatments=[]
    if not pd.isna(row['mtx_dose']) and row['mtx_dose'] > 0:
        treatments.append(f"MTX at {row['mtx_dose']:.0f} mg/week")
    if not pd.isna(row['steroid_dose']) and row['steroid_dose'] > 0:
        treatments.append(f"steroids at {row['steroid_dose']:.0f} mg/day")
    if not pd.isna(row['csdmard_use']) and row['csdmard_use'] == 1:
        treatments.append("a csDMARD")
    if not pd.isna(row['b_ts_dmard_use']) and row['b_ts_dmard_use'] == 1:
        treatments.append("a b/tsDMARD")

    treatment_sentence= 'The patient is currently receiving ' + ', '.join(treatments) +'. '\
        if treatments else "The patient is not currently on any DMARD or steroid therapy."
    decision = row['treatment_decision']
    treatment_sentence += f"The treatment decision at this visit was: {decision}." 
    parts.append(treatment_sentence)

    #History
    prev='had a flare' if (not pd.isna(row['previous_flare']) and row['previous_flare']==1) else 'did not have a flare' if (not pd.isna(row['previous_flare']) and row['previous_flare']==0) else 'flare history unknown'
    if is_first_visit(row):
        parts.append(f"This is the patient's first recorded visit. Information about the previous flare is not available. ")
                     
    else:
        parts.append(f"This is a follow-up visit, {row['days_since_last_visit']} days since the last assessment. "
                     f"At the previous visit, the patient {prev}.")
        
    #Comorbidities
    comorbidities_text = get_comorbidities_text(row['comorbidities'])
    parts.append(f"Comorbidities: {comorbidities_text}.")

    return ' '.join(parts)
print(serialize_text(df_llm.iloc[5]))

In [ ]:
client = anthropic.Anthropic(api_key="ANTHROPIC_API_KEY")

In [ ]:
SYSTEM_PROMPT= """
You are an expert rheumatologist specializing in Rheumatoid Arthritis. You will be given a clinical visit summary and asked to predict whether the patient will experience a disease flare at their next visit.

In this specific clinical cohort, approximately 31% of visits are followed by a disease flare at the next appointment. The majority of patients (69%) remain stable between visits.

A disease flare is a clinically meaningful increase in disease activity at the subsequent visit, characterised by worsening joint symptoms and elevated inflammatory markers.

You must weigh risk factors against protective factors carefully. Do not default to predicting flare unless the clinical evidence clearly supports it.

Structure your response as:
<prediction>Yes</prediction> or <prediction>No</prediction>

Write only one prediction tag, and do not include any text outside the tags.

"""

In [ ]:
#Strategy 1: Zero-shot prompt
def build_zero_shot_prompt(row, serializer):
    visit_summary = serializer(row)
    return f"""Clinical Visit Summary:
{visit_summary}

Based on this clinical profile, will this patient experience a disease flare at their next visit?
Return your answer exactly as:
<prediction>Yes</prediction> or <prediction>No</prediction>"""

In [ ]:
#Strategy 2: Few-shot prompt
def build_few_shot_prompt(row, examples, serializer=serialize_list):
    prompt = f"Here are {len(examples)} example cases with known outcomes:\n\n"
    
    for i, (ex_row, ex_label) in enumerate(examples, 1):
        label_text = "Yes" if ex_label == 1 else "No"
        prompt += f"Example {i}:\n"
        prompt += serializer(ex_row)
        prompt += f"\n<prediction>{label_text}</prediction>\n\n---\n\n"
    prompt += "Now predict for this new patient:\n\n"
    prompt += "Clinical Visit Summary:\n\n"
    prompt += serializer(row)
    prompt += "\n\nProvide your prediction using the XML format specified in the system instructions."
    
    return prompt


In [ ]:
def parse_prediction(response):
    """Extract binary prediction from model response."""
    if response is None:
        return -1

    match = re.search(r"<prediction>\s*(Yes|No)\s*</prediction>", response, re.IGNORECASE)
    if match:
        return 1 if match.group(1).lower() == 'yes' else 0

    tagged_final = re.search(r"(?:^|\n)\s*(?:final answer|prediction)\s*[:]?\s*(yes|no)\s*(?:$|\n)", response, re.IGNORECASE)
    if tagged_final:
        return 1 if tagged_final.group(1).lower() == 'yes' else 0

    tail = response.strip().splitlines()[-3:]
    tail_text = ' '.join(tail)
    tail_match = re.search(r"\b(yes|no)\b", tail_text, re.IGNORECASE)
    if tail_match:
        return 1 if tail_match.group(1).lower() == 'yes' else 0

    print(f"Unparseable response: {response[:100]}")
    return -1


def extract_reasoning(response):
    """Extract clinical reasoning text for qualitative analysis."""
    if response is None:
        return None

    match = re.search(r'<clinical_reasoning>(.*?)</clinical_reasoning>', response, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()

    fallback = re.search(r'<clinical_reasoning>(.*?)(?:<prediction>|$)', response, re.DOTALL | re.IGNORECASE)
    return fallback.group(1).strip() if fallback else None

In [ ]:
def query_claude(user_prompt, system_prompt=SYSTEM_PROMPT, model="claude-haiku-4-5-20251001", max_tokens=30):
    try:
        message = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system_prompt,
            temperature=0,
            messages=[{"role": "user", "content": user_prompt}]
        )
        return message.content[0].text.strip()
    
    except anthropic.RateLimitError:
        print('Rate limit hit. Waiting 15 seconds before retrying...')
        time.sleep(15)
        return query_claude(user_prompt, system_prompt, model, max_tokens)
    except Exception as e:
        print(f"Error querying Claude: {e}")
        return None


def run_cv_classification(df_llm, prompt_builder, strategy_name, serializer, save_every=50, delay=0.5):
    stratified_group_kfold=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

    groups=df_llm['patient_id'].values
    y=df_llm['flare_next_visit'].values
    filename = f'cv_results_{strategy_name.replace(" ", "_").lower()}.csv'

    all_results = []
    print(f"Running cross-validation with strategy: {strategy_name}")
    print(f"Total samples: {len(df_llm)}")
    
    for fold_idx, (train_idx, test_idx) in enumerate(stratified_group_kfold.split(df_llm, y, groups)):
        print(f"\n---Processing Fold {fold_idx + 1} / 5 ---")
        df_train_fold = df_llm.iloc[train_idx]
        df_test_fold = df_llm.iloc[test_idx]

        fold_few_shot_examples = []
        if 'few' in strategy_name.lower() and 'shot' in strategy_name.lower():
            flare_pool=df_train_fold[
                                (df_train_fold['flare_next_visit'] == 1) &
                                (df_train_fold['das28_score'].between(4, 6.5)) &
                                (df_train_fold['previous_flare'] == 1) &
                                ((df_train_fold['crp'] > df_train_fold['crp_upper_limit']) | (df_train_fold['esr'] > 20))
                                ].dropna(subset=['das28_score', 'crp', 'esr'])
            
            no_flare_pool=df_train_fold[
                                (df_train_fold['flare_next_visit'] == 0) &
                                (df_train_fold['das28_score'].between(2.6, 4)) &
                                (df_train_fold['previous_flare'] == 0) &
                                ((df_train_fold['crp'] <= df_train_fold['crp_upper_limit']) & (df_train_fold['esr'] <= 20))
                                ].dropna(subset=['das28_score', 'crp', 'esr'])
            
            few_shot_flare = flare_pool.drop_duplicates(subset=['patient_id']).sample(n=3, random_state=42)
            few_shot_no_flare = no_flare_pool.drop_duplicates(subset=['patient_id']).sample(n=3, random_state=42)
            
            fold_few_shot_examples = (
                [(row, 1) for _, row in few_shot_flare.iterrows()] +
                [(row, 0) for _, row in few_shot_no_flare.iterrows()]
            )

        for i, (idx, row) in enumerate(tqdm(df_test_fold.iterrows(), total=len(df_test_fold), desc=f"Fold {fold_idx+1}")):
            if 'few' in strategy_name.lower() and 'shot' in strategy_name.lower():
                prompt = prompt_builder(row, fold_few_shot_examples, serializer)
            else:
                prompt = prompt_builder(row, serializer)

            response = query_claude(prompt)
            prediction = parse_prediction(response)
            actual = row['flare_next_visit']

            all_results.append({
                'fold': fold_idx + 1,
                'visit_idx': idx,
                'patient_id': row['patient_id'],
                'actual_flare': actual,
                'predicted_flare': prediction,
                'correct': (prediction == actual) if prediction != -1 else None,
                'das28': row['das28_score'],
                'response': response,
                'serializer': serializer.__name__,
                'strategy': strategy_name
            })

            if (len(all_results)) % save_every == 0:
                pd.DataFrame(all_results).to_csv(filename, index=False)
            
            time.sleep(delay)

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(filename, index=False)

    return results_df


def evaluate_results(results_df, strategy_name):
    valid=results_df[results_df['predicted_flare'] != -1].copy()
    unparseable = (results_df['predicted_flare'] == -1).sum()

    y_true=valid['actual_flare'].values.astype(int)
    y_pred=valid['predicted_flare'].values.astype(int)

    print(f"\n--- Evaluation for {strategy_name} ---")
    print(f"Total visits: {len(results_df)}")
    print(f"Valid predictions: {len(valid)}")
    print(f"Unparseable responses: {unparseable}")

    if len(valid) == 0:
        print("No valid predictions to evaluate.")
        return None
    
    report = classification_report(y_true, y_pred, target_names=['No Flare', 'Flare'], zero_division=0)
    print(f"\n{report}")

    metrics={}

    if len(np.unique(y_pred)) > 1:
        metrics['average_precision'] = average_precision_score(y_true, y_pred)
        metrics['roc_auc'] = roc_auc_score(y_true, y_pred)
        print(f"Average Precision: {metrics['average_precision']:.3f}")
        print(f"ROC AUC: {metrics['roc_auc']:.3f}")
    else:
        print("Only one class predicted; cannot compute Average Precision or ROC AUC.")
        print(f"Predicted classes: {np.unique(y_pred)[0]}")
        metrics['average_precision'] = None
        metrics['roc_auc'] = None
    
    print("\nConfusion Matrix:")
    cm=confusion_matrix(y_true, y_pred)
    print(cm)
    print(f" TN={cm[0,0]}, FP={cm[0,1]}")
    print(f" FN={cm[1,0]}, TP={cm[1,1]}")

    metrics.update({
        'strategy': strategy_name,
        'total_visits': len(results_df),
        'valid_predictions': len(valid),
        'unparseable_responses': unparseable,
        'recall_flare': cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0,
        'precision_flare': cm[1,1]/(cm[0,1]+cm[1,1]) if (cm[0,1]+cm[1,1])>0 else 0,
    })

    return metrics

In [ ]:
#Zero-Shot + List template
results_zs_list =run_cv_classification(df_llm, build_zero_shot_prompt, "Zero-Shot + List Template", serializer=serialize_list)
metrics_zs_list = evaluate_results(results_zs_list, "Zero-Shot + List Template")

In [ ]:
#Zero-shot + Text template
results_zs_text = run_cv_classification(df_llm, build_zero_shot_prompt, "Zero-Shot + Text Template", serializer=serialize_text)
metrics_zs_text = evaluate_results(results_zs_text, "Zero-Shot + Text Template")

In [ ]:
#Few-shot + List template
results_fs_list = run_cv_classification(df_llm, build_few_shot_prompt, "Few-Shot + List Template", serializer=serialize_list)
metrics_fs_list = evaluate_results(results_fs_list, "Few-Shot + List Template")

In [ ]:
#Few-shot + Text template
results_fs_text = run_cv_classification(df_llm, build_few_shot_prompt, "Few-Shot + Text Template", serializer=serialize_text)
metrics_fs_text = evaluate_results(results_fs_text, "Few-Shot + Text Template")

## Conclusions
All four LLM configurations underperformed the tuned
Logistic Regression on both primary metrics:

| Approach | AP | ROC-AUC |
|---|---|---|
| Logistic Regression | 0.508 | 0.695 |
| Few-shot / List (best LLM) | 0.42 | 0.64 |
| Zero-shot / List | 0.41 | 0.63 |
| Zero-shot / Text | 0.40 | 0.62 |
| Few-shot / Text | 0.40 | 0.61 |

### Three Key Observations

**1. ML outperforms LLM consistently**
The AP gap ranges from 0.088 to 0.108 between Logistic Regression and the best LLM configuration.

**2. Few-shot improves over zero-shot (but only for the list template)**
Few-shot prompting with the structured list template improved AP from 0.41 to 0.42 and ROC-AUC from
0.63 to 0.64. However, few-shot with the narrative text template degraded performance — suggesting
clinical examples interact with serialization format in ways that require careful prompt design.

**3. Serialization format has negligible impact
on zero-shot performance**
List versus text template differed by less than 0.01 AP under zero-shot conditions meaning information
content drives LLM performance, not presentation format.

### Clinical Interpretation
The LLM adopted a more conservative prediction
strategy than the ML models. It achieved higher
specificity (0.61–0.63 vs 0.48 for LR) but
substantially lower flare recall (0.61–0.64 vs 0.83).
This reflects asymmetric risk reasoning encoded in general medical training data rather than model failure.

### Conclusion
General medical knowledge encoded during LLM pretraining
is insufficient to substitute for patient-specific
longitudinal supervised training on this task.
However, few-shot configurations offer a viable
training-free baseline in data-scarce settings
where supervised training is not feasible — accepting
lower recall in exchange for training-free deployment.